# Black-Scholes Option Pricer — Demo / デモ

**EN.** This notebook demonstrates the `src.pricing` package: the closed-form
Black-Scholes price and Greeks, the implied-volatility solver, Monte Carlo
convergence to the analytic price, and a simple volatility smile.

**JA.** このノートブックは `src.pricing` パッケージのデモです。Black-Scholes の解析解と
Greeks、インプライド・ボラティリティの逆算、モンテカルロ法の解析解への収束、そして
簡単なボラティリティ・スマイルを可視化します。

In [ ]:
# Make the project root importable so `import src...` works from notebooks/.
# プロジェクトルートを import パスに追加します。
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.pricing.black_scholes import (
    bs_price,
    delta,
    gamma,
    vega,
    theta,
    rho,
    greeks,
    implied_volatility,
    OptionType,
)
from src.pricing.monte_carlo import mc_price

sns.set_theme(style="whitegrid", context="notebook")

# Base market scenario / 基本シナリオ
S, K, T, r, sigma = 100.0, 100.0, 1.0, 0.05, 0.20
print("Base call price:", round(bs_price(S, K, T, r, sigma, OptionType.CALL), 4))
print("Base put price :", round(bs_price(S, K, T, r, sigma, OptionType.PUT), 4))

## 1. Price vs Spot / 価格と原資産価格

**EN.** Call and put premiums as a function of spot, with the intrinsic-value
payoff for reference. **JA.** 原資産価格に対するコール/プットのプレミアム。点線は満期
ペイオフ（本質的価値）です。

In [ ]:
spots = np.linspace(40, 160, 200)
call = bs_price(spots, K, T, r, sigma, OptionType.CALL)
put = bs_price(spots, K, T, r, sigma, OptionType.PUT)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(spots, call, label="Call price")
ax.plot(spots, put, label="Put price")
ax.plot(spots, np.maximum(spots - K, 0), "--", alpha=0.6, label="Call intrinsic")
ax.plot(spots, np.maximum(K - spots, 0), "--", alpha=0.6, label="Put intrinsic")
ax.axvline(K, color="grey", lw=0.8, alpha=0.5)
ax.set(xlabel="Spot S", ylabel="Option value", title="Black-Scholes price vs spot")
ax.legend()
plt.tight_layout()
plt.show()

## 2. Greeks vs Spot / グリークスと原資産価格

**EN.** The five Greeks for a call across spot. Delta runs 0→1, Gamma peaks
near the money, Vega is largest ATM. **JA.** コールの5つの Greeks。Delta は 0→1、
Gamma と Vega はアットザマネー付近で最大になります。

In [ ]:
greek_funcs = {
    "Delta": delta(spots, K, T, r, sigma, OptionType.CALL),
    "Gamma": gamma(spots, K, T, r, sigma),
    "Vega": vega(spots, K, T, r, sigma),
    "Theta (per day)": theta(spots, K, T, r, sigma, OptionType.CALL),
    "Rho": rho(spots, K, T, r, sigma, OptionType.CALL),
}

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, (name, values) in zip(axes.flat, greek_funcs.items()):
    ax.plot(spots, values)
    ax.axvline(K, color="grey", lw=0.8, alpha=0.5)
    ax.set(title=name, xlabel="Spot S")
axes.flat[-1].axis("off")  # only five Greeks for a 2x3 grid
fig.suptitle("Call Greeks vs spot", fontsize=14)
plt.tight_layout()
plt.show()

## 3. Monte Carlo convergence / モンテカルロの収束

**EN.** As the number of paths grows, the MC estimate converges to the analytic
price and the 95% confidence band narrows as $1/\sqrt{n}$. **JA.** パス数を増やすと
MC 推定値は解析解に収束し、95% 信頼区間は $1/\sqrt{n}$ で狭まります。

In [ ]:
analytic = bs_price(S, K, T, r, sigma, OptionType.CALL)
path_counts = [1_000, 2_000, 5_000, 10_000, 20_000, 50_000, 100_000, 200_000]

prices, lows, highs = [], [], []
for n in path_counts:
    res = mc_price(S, K, T, r, sigma, OptionType.CALL, n_paths=n, seed=2024)
    prices.append(res.price)
    lows.append(res.ci_95[0])
    highs.append(res.ci_95[1])

fig, ax = plt.subplots(figsize=(9, 5))
ax.fill_between(path_counts, lows, highs, alpha=0.25, label="95% CI")
ax.plot(path_counts, prices, "o-", label="MC estimate")
ax.axhline(analytic, color="red", ls="--", label=f"Analytic = {analytic:.4f}")
ax.set_xscale("log")
ax.set(xlabel="Number of paths (log)", ylabel="Call price",
       title="Monte Carlo convergence to the Black-Scholes price")
ax.legend()
plt.tight_layout()
plt.show()

## 4. Volatility smile / ボラティリティ・スマイル

**EN.** Black-Scholes assumes a single flat vol, so inverting *its own* prices
returns a flat line. To illustrate a smile we add a stylised skew to the input
vols, price with it, then back out implied vol per strike — recovering the smile.

**JA.** Black-Scholes は単一のフラットなボラを仮定するため、自身の価格を逆算すると
直線になります。スマイルを示すため、入力ボラに作為的なスキューを加えて価格付けし、
ストライクごとにインプライド・ボラを逆算してスマイルを再現します。

In [ ]:
strikes = np.linspace(70, 130, 25)
moneyness = strikes / S

# Stylised smile: vol rises away from the money (a convex curve in log-moneyness).
# 作為的スマイル: アットザマネーから離れるほどボラが上昇する凸カーブ。
base_vol, curvature = 0.20, 0.6
input_vols = base_vol + curvature * (np.log(moneyness)) ** 2

market_prices = [bs_price(S, k, T, r, v, OptionType.CALL)
                 for k, v in zip(strikes, input_vols)]
implied = [implied_volatility(p, S, k, T, r, OptionType.CALL)
           for p, k in zip(market_prices, strikes)]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(strikes, np.array(implied) * 100, "o-", label="Recovered implied vol")
ax.axvline(S, color="grey", lw=0.8, alpha=0.5, label="ATM")
ax.set(xlabel="Strike K", ylabel="Implied volatility (%)",
       title="Volatility smile recovered by the IV solver")
ax.legend()
plt.tight_layout()
plt.show()

max_err = np.max(np.abs(np.array(implied) - input_vols))
print(f"Max |recovered - input| vol error: {max_err:.2e}")

## 5. Risk report / リスク・レポート

**EN.** The `greeks()` helper returns price plus all sensitivities in one call.
**JA.** `greeks()` は価格と全感応度を一度に返します。

In [ ]:
report = greeks(S, K, T, r, sigma, OptionType.CALL)
for name, value in report.items():
    print(f"{name:>6}: {value:>12.6f}")